# 03 — Phase 2: GQA + PagedAttention (vLLM)

**Framework**: vLLM — the most widely supported PagedAttention implementation

### Why not implement PagedAttention from scratch?
True paging is difficult to achieve with native PyTorch. vLLM provides:
- **Custom CUDA kernels**: attention computed directly on block-fragmented memory
- **On-demand block allocation**: eliminates fragmentation from pre-allocation
- **Unified memory management**: block table tracks physical→virtual mappings

### Expected Outcome
- TTFT / TPOT comparable to baseline (same computation)
- **Dramatically lower memory fragmentation**
- **Higher memory utilisation** → same 10–12 GB can handle longer sequences

### Install vLLM (Jetson)
```bash
# See scripts/setup_vllm_jetson.sh
pip install vllm  # or build from source
```

⚠️ If vLLM is not installed, this notebook falls back to a HF-native fragmentation comparison demo.

In [ ]:
import sys, gc, time, torch
sys.path.insert(0, '..')

from src.vllm_runner import (
    check_vllm_available, create_vllm_engine,
    run_vllm_benchmark, get_vllm_cache_stats,
)
from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.paged_cache import PagedKVCache
from src.dataset_utils import load_prompts

USE_VLLM = check_vllm_available()
print(f"vLLM available: {USE_VLLM}")
if not USE_VLLM:
    print("\n⚠️  vLLM not installed. Will demonstrate paging concepts with the Python reference implementation.")
    print("    Real PagedAttention performance gains require vLLM CUDA kernels.")
    print("    Installation: see scripts/setup_vllm_jetson.sh")

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


INFO 04-23 21:29:08 [__init__.py:239] Automatically detected platform cuda.


2026-04-23 21:29:13,056	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


vLLM available: True


In [2]:
prompts = load_prompts('../results/ehr_prompts_llama3.2.json')
print(f"Loaded {len(prompts)} prompts")

Loaded 50 prompts


---
## Path A: vLLM Engine (True PagedAttention)

In [3]:
if USE_VLLM:
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    
    engine = create_vllm_engine(
        model_name=MODEL_NAME,
        gpu_memory_utilization=0.90,
        max_model_len=4096,
        block_size=16,
    )
    
    # PagedAttention block 统计
    cache_stats = get_vllm_cache_stats(engine)
    print(f"PagedAttention stats:")
    for k, v in cache_stats.items():
        print(f"  {k}: {v}")
else:
    print("Skipping vLLM engine creation (not installed)")

WARNING 04-23 21:29:14 [config.py:3002] Casting torch.bfloat16 to torch.float16.
INFO 04-23 21:29:31 [config.py:730] This model supports multiple tasks: {'score', 'classify', 'generate', 'reward', 'embed'}. Defaulting to 'generate'.
INFO 04-23 21:29:31 [config.py:2016] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-23 21:29:31 [cuda.py:93] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 04-23 21:29:35 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized


/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


INFO 04-23 21:29:43 [__init__.py:239] Automatically detected platform cuda.
INFO 04-23 21:29:49 [core.py:58] Initializing a V1 LLM engine (v0.8.6) with config: model='unsloth/Llama-3.2-3B-Instruct', speculative_config=None, tokenizer='unsloth/Llama-3.2-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=None, served_model_name=unsloth/Llama-3.2-3B-Instruct, num_scheduler_steps=1, multi_s

[W423 21:29:51.626411746 ProcessGroupNCCL.cpp:959] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 04-23 21:29:51 [parallel_state.py:1004] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0
INFO 04-23 21:29:51 [cuda.py:221] Using Flash Attention backend on V1 engine.
INFO 04-23 21:29:51 [topk_topp_sampler.py:59] Using FlashInfer for top-p & top-k sampling.
INFO 04-23 21:29:51 [gpu_model_runner.py:1329] Starting to load model unsloth/Llama-3.2-3B-Instruct...
INFO 04-23 21:29:54 [weight_utils.py:265] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.31s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:06<00:00,  3.31s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:06<00:00,  3.01s/it]



INFO 04-23 21:30:01 [loader.py:458] Loading weights took 6.29 seconds
INFO 04-23 21:30:01 [gpu_model_runner.py:1347] Model loading took 6.0160 GiB and 9.589976 seconds
INFO 04-23 21:30:13 [kv_cache_utils.py:634] GPU KV cache size: 22,192 tokens
INFO 04-23 21:30:13 [kv_cache_utils.py:637] Maximum concurrency for 4,096 tokens per request: 5.42x
INFO 04-23 21:30:16 [core.py:159] init engine (profile, create kv cache, warmup model) took 15.24 seconds
INFO 04-23 21:30:16 [core_client.py:439] Core engine process 0 ready.

✅ vLLM Engine Initialized
  Model          : unsloth/Llama-3.2-3B-Instruct
  Dtype          : float16
  Max Seq Len    : 4096
  Block Size     : 16
  Cache Dtype    : auto
  GPU Mem Util   : 90%

📊 PagedAttention Config:
  Num GPU Blocks : None
  Block Size     : 16

PagedAttention stats:
  num_total_gpu_blocks: None
  num_free_gpu_blocks: 0
  block_size: 16


In [ ]:
# --- vLLM PagedAttention OOM threshold probe ---
# The GQA-only baseline OOMs at ctx≈2048; test whether vLLM handles longer contexts
if USE_VLLM:
    from src.vllm_runner import find_vllm_oom_threshold

    print("vLLM PagedAttention OOM threshold probe ...")
    oom_result = find_vllm_oom_threshold(
        engine,
        context_lengths=[256, 512, 1024, 2048, 4096, 8192],
        max_new_tokens=32,
    )

    print(f"\nMax safe length: {oom_result['max_safe_length']} tokens")
    if oom_result["oom_length"]:
        print(f"OOM triggered at: {oom_result['oom_length']} tokens")
    else:
        print("All tested lengths passed — no OOM triggered")

    # Per-length results
    for r in oom_result["results"]:
        status = "PASS" if r["success"] else "FAIL"
        print(f"  {status} ctx={r['context_length']:>5d}  {r.get('error', 'OK')}")
else:
    print("Skipping OOM probe (vLLM not installed)")

vLLM PagedAttention OOM 阈值探测 ...


Processed prompts:   0%|                                                                            | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
if USE_VLLM:
    results_paged = run_vllm_benchmark(
        engine, prompts,
        max_new_tokens=256,
        warmup_runs=2,
    )
    
    import pandas as pd
    df_vllm = pd.DataFrame(results_paged)
    df_vllm.to_csv('../results/gqa_paged_vllm.csv', index=False)
    
    cols = ['ttft_ms', 'tpot_ms', 'total_time_ms', 'num_input_tokens', 'num_output_tokens']
    print(df_vllm[cols].describe().round(1))
else:
    print("Skipping vLLM benchmark")

---
## Batch Size Comparison

The above runs BS=1 (sequential single requests).
Below we add throughput comparisons for BS=2 / 4 / 8.
vLLM's scheduler only shows its full advantage in batched scenarios.

In [ ]:
if USE_VLLM:
    batch_sizes = [2, 4]  # On Jetson 16 GB, keep to 4–8 max depending on available memory
    
    for bs in batch_sizes:
        print(f"\n{'='*50}")
        print(f"▶ vLLM Batch Benchmark  BS={bs}")
        print(f"{'='*50}")
        
        results_batch = run_vllm_benchmark(
            engine, prompts,
            max_new_tokens=256,
            warmup_runs=2,
            batch_size=bs,
        )
        
        df_batch = pd.DataFrame(results_batch)
        df_batch.to_csv(f'../results/gqa_paged_vllm_bs{bs}.csv', index=False)
        
        # Throughput estimate: total output tokens / total wall-clock time
        total_out_tokens = df_batch['num_output_tokens'].sum()
        total_time_sec = df_batch['total_time_ms'].sum() / 1000.0
        throughput = total_out_tokens / total_time_sec if total_time_sec > 0 else 0
        
        print(f"\nBS={bs} Summary:")
        print(df_batch[cols].describe().round(1))
        print(f"\nThroughput estimate: {throughput:.1f} tokens/sec "
              f"({total_out_tokens} output tokens / {total_time_sec:.1f} sec)")
else:
    print("Skipping batch benchmark (vLLM not installed)")

---
## PPL Evaluation

PagedAttention is a **lossless** memory-management optimisation — it does not alter numerical results.
PPL should equal the GQA-only baseline. This cell verifies that assumption.

In [ ]:
from src.perplexity import compute_perplexity
from src.dataset_utils import load_prompts
import json, pandas as pd

# PagedAttention does not change numerical results — use HF model for PPL verification.
# Reuse 'model' if already loaded from Path B; otherwise load it now.
if 'model' not in dir():
    from src.jetson_utils import load_model_safe
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    FALLBACK_NAME = "unsloth/Llama-3.2-3B-Instruct"
    model, tokenizer = load_model_safe(MODEL_NAME, fallback_name=FALLBACK_NAME, device="cuda")

if 'prompts' not in dir() or not prompts:
    prompts = load_prompts('../results/ehr_prompts_llama3.json')

eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]
print(f"PPL evaluation on {len(eval_texts)} texts...")

ppl_paged = compute_perplexity(model, tokenizer, eval_texts, max_length=512)
print(f"\nPagedAttn PPL : {ppl_paged['ppl']:.2f}")
print(f"Avg loss      : {ppl_paged['avg_loss']:.4f}")
print(f"Tokens eval   : {ppl_paged['num_tokens']}")

# Compare against baseline
ppl_path = '../results/ppl_results.json'
try:
    with open(ppl_path) as f:
        ppl_all = json.load(f)
    baseline_ppl = ppl_all['GQA_only']['ppl']
    degradation = (ppl_paged['ppl'] - baseline_ppl) / baseline_ppl * 100
    print(f"\nBaseline PPL  : {baseline_ppl:.2f}")
    print(f"Degradation   : {degradation:+.1f}% (expected ≈ 0%)")
except FileNotFoundError:
    ppl_all = {}
    print("⚠️ Baseline PPL file not found — run notebook 02 first")

# Save
ppl_all['GQA+PagedAttn'] = ppl_paged
with open(ppl_path, 'w') as f:
    json.dump(ppl_all, f, indent=2)

ppl_df = pd.DataFrame([{
    'config': 'GQA+PagedAttn',
    'ppl': ppl_paged['ppl'],
    'avg_loss': ppl_paged['avg_loss'],
    'num_tokens': ppl_paged['num_tokens'],
}])
ppl_df.to_csv('../results/ppl_gqa_paged.csv', index=False)
print("Saved → results/ppl_results.json + results/ppl_gqa_paged.csv")

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:23<00:00, 11.73s/it]


✓ Loaded unsloth/Llama-3.2-3B-Instruct (torch.float16)  GPU mem: 5.98 GB
PPL evaluation on 20 texts...


Computing PPL: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.11it/s]



PagedAttn PPL : 45.71
Avg loss      : 3.8222
Tokens eval   : 253

Baseline PPL  : 45.71
退化          : +0.0% (预期 ≈ 0%)
Saved → results/ppl_results.json + results/ppl_gqa_paged.csv


In [ ]:
# 释放模型（后续 notebook 重新加载）
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")